In [ ]:
import pandas as pd

raw_data = {
    "transaction_id": range(1, 26),
    "customer_id": [201,202,203,201,204,202,203,205,201,202,
                    204,203,201,205,202,204,203,201,202,205,
                    204,203,201,202,204],
    "product_name": ["Laptop","Headphones","Mouse","Laptop","Keyboard",
                     "Monitor","Headphones","Mouse","Keyboard","Laptop",
                     "Monitor","Mouse","Laptop","Keyboard","Headphones",
                     "Laptop","Monitor","Mouse","Keyboard","Laptop",
                     "Headphones","Keyboard","Monitor","Mouse","Laptop"],
    "category": ["Electronics","Electronics","Electronics","Electronics","Electronics",
                 "Electronics","Electronics","Electronics","Electronics","Electronics",
                 "Electronics","Electronics","Electronics","Electronics","Electronics",
                 "Electronics","Electronics","Electronics","Electronics","Electronics",
                 "Electronics","Electronics","Electronics","Electronics","Electronics"],
    "unit_price": [8500000,350000,125000,8500000,275000,
                   3200000,350000,125000,275000,8500000,
                   3200000,125000,8500000,275000,350000,
                   8500000,3200000,125000,275000,8500000,
                   350000,275000,3200000,125000,8500000],
    "quantity": [1,2,3,1,2,1,1,4,2,1,1,2,1,3,2,1,1,5,2,1,2,1,1,3,1],
    "status": ["completed","completed","completed","completed","cancelled",
               "completed","completed","completed","completed","cancelled",
               "completed","completed","completed","completed","completed",
               "cancelled","completed","completed","completed","completed",
               "completed","completed","completed","completed","completed"],
    "transaction_date": [
        "2024-01-03","2024-01-05","2024-01-07","2024-01-10","2024-01-12",
        "2024-01-15","2024-01-18","2024-01-20","2024-02-02","2024-02-05",
        "2024-02-08","2024-02-10","2024-02-14","2024-02-18","2024-02-20",
        "2024-02-25","2024-03-01","2024-03-05","2024-03-08","2024-03-12",
        "2024-03-15","2024-03-18","2024-03-20","2024-03-22","2024-03-25"
    ]
}

pd.DataFrame(raw_data).to_csv("raw_transactions.csv", index=False)

In [ ]:
# Task 1 — Load & Clean
# Baca file raw_transactions.csv, lalu:

# Parse transaction_date sebagai datetime saat read (hint: parse_dates parameter di read_csv)
# Filter hanya completed
# Tambahkan kolom total_amount = unit_price × quantity
# Tambahkan kolom month format YYYY-MM
# Tambahkan kolom week_type: "weekday" atau "weekend"

raw_transactions = pd.read_csv("raw_transactions.csv", parse_dates=['transaction_date'])
completed_trx = raw_transactions[raw_transactions["status"] == "completed"].copy()
completed_trx["total_amount"] = completed_trx["unit_price"] * completed_trx["quantity"]
completed_trx["month"] = completed_trx["transaction_date"].dt.strftime("%Y-%m")
completed_trx["week_type"] = completed_trx["transaction_date"].dt.day_of_week >= 5

completed_trx["week_type"] = completed_trx["week_type"].apply(
    lambda x: "weekday" if x == False else "weekend"
)
completed_trx

,transaction_id,customer_id,product_name,category,unit_price,quantity,status,transaction_date,total_amount,month,week_type
0,1,201,Laptop,Electronics,8500000,1,completed,2024-01-03,8500000,2024-01,weekday
1,2,202,Headphones,Electronics,350000,2,completed,2024-01-05,700000,2024-01,weekday
2,3,203,Mouse,Electronics,125000,3,completed,2024-01-07,375000,2024-01,weekend
3,4,201,Laptop,Electronics,8500000,1,completed,2024-01-10,8500000,2024-01,weekday
5,6,202,Monitor,Electronics,3200000,1,completed,2024-01-15,3200000,2024-01,weekday
6,7,203,Headphones,Electronics,350000,1,completed,2024-01-18,350000,2024-01,weekday
7,8,205,Mouse,Electronics,125000,4,completed,2024-01-20,500000,2024-01,weekend
8,9,201,Keyboard,Electronics,275000,2,completed,2024-02-02,550000,2024-02,weekday
10,11,204,Monitor,Electronics,3200000,1,completed,2024-02-08,3200000,2024-02,weekday
11,12,203,Mouse,Electronics,125000,2,completed,2024-02-10,250000,2024-02,weekend


In [ ]:

# Task 2 — Monthly Product Report
# Buat pivot table:
# month | Headphones | Keyboard | Laptop | Monitor | Mouse
# Nilai = total total_amount per produk per bulan. Ganti NaN dengan 0. Tambahkan kolom monthly_total yang berisi sum semua produk per bulan.
# Simpan output ke data/monthly_product_report.csv.

monthly_product_report = completed_trx.pivot_table(
    index="month",
    columns="category",
    values="total_amount",
    aggfunc="sum",
    fill_value=0
)

monthly_product_report["monthly_total"] = monthly_product_report.sum(axis=1)

monthly_product_report.to_csv("monthly_product_report.csv", index=False)

category,Electronics,monthly_total
month,,
2024-01,22125000,62075000
2024-02,14025000,62075000
2024-03,25925000,62075000


In [ ]:
completed_trx.head(1)

,transaction_id,customer_id,product_name,category,unit_price,quantity,status,transaction_date,total_amount,month,week_type
0,1,201,Laptop,Electronics,8500000,1,completed,2024-01-03,8500000,2024-01,weekday


In [ ]:

# Task 3 — Customer Summary
# Buat report per customer:
# customer_id | total_spent | total_trx | avg_order |
# first_purchase | last_purchase | customer_lifetime_days | vip_status

# customer_lifetime_days = selisih hari antara first_purchase dan last_purchase
# vip_status: "VIP" kalau total_spent >= 10.000.000, selain itu "Regular"

# Simpan ke data/customer_summary.parquet.

customer_summary = completed_trx.groupby("customer_id").agg(
    total_spent = ("total_amount", "sum"),
    total_trx = ("transaction_id", "count"),
    avg_order = ("quantity", "mean"),
    first_purchase = ("transaction_date", "min"),
    last_purchase = ("transaction_date", "max")
)

customer_summary["customer_lifetime_days"] = (customer_summary["last_purchase"] - customer_summary["first_purchase"]).dt.days
customer_summary["vip_status"] = customer_summary["total_spent"].apply(
    lambda x: "vip" if x >= 10000000 else "Regular"
)
customer_summary.to_parquet("customer_summary.parquet")

,total_spent,total_trx,avg_order,first_purchase,last_purchase,customer_lifetime_days,vip_status
customer_id,,,,,,,
201,29875000,6,1.833333,2024-01-03,2024-03-20,77,vip
202,5525000,5,2.000000,2024-01-05,2024-03-22,77,Regular
203,4450000,5,1.600000,2024-01-07,2024-03-18,71,Regular
204,12400000,3,1.333333,2024-02-08,2024-03-25,46,vip
205,9825000,3,2.666667,2024-01-20,2024-03-12,52,Regular
